# Train A Piper Voice In Colab

This notebook is the public quick-start path for the repository.

Workflow:

1. mount Google Drive
2. point the notebook at a reviewed dataset
3. train in Colab
4. export ONNX
5. copy the results back to Drive

## 0. Check Colab GPU

In [ ]:
import sys
import torch

print("Python:", sys.version.split()[0])
print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
!nvidia-smi


## 1. Install system packages

In [ ]:
!sudo apt-get update -y
!sudo apt-get install -y build-essential cmake ninja-build espeak-ng espeak-ng-data libespeak-ng-dev pkg-config ffmpeg
!pkg-config --modversion espeak-ng


## 2. Clone Piper and install training dependencies

In [ ]:
%cd /content
!rm -rf piper1-gpl
!git clone https://github.com/OHF-Voice/piper1-gpl.git
%cd /content/piper1-gpl
!python3 -m pip install --upgrade pip setuptools wheel scikit-build cmake ninja
!python3 -m pip install -e ".[train]"
!chmod +x ./build_monotonic_align.sh
!./build_monotonic_align.sh


## 3. Mount Google Drive

In [ ]:
from google.colab import drive

drive.mount("/content/drive")


## 4. Set project paths and training values

Edit this cell only. Everything below uses these variables.

In [ ]:
from pathlib import Path

VOICE_NAME = "my_voice"
ESPEAK_VOICE = "ru"
SAMPLE_RATE_HZ = 22050
BATCH_SIZE = 8
MAX_EPOCHS = 6000
BASE_CHECKPOINT = ""

PROJECT_DIR = Path("/content/drive/MyDrive/tts_projects") / VOICE_NAME
METADATA_PATH = PROJECT_DIR / "metadata.csv"
AUDIO_DIR = PROJECT_DIR / "wavs"
CONFIG_PATH = PROJECT_DIR / f"{VOICE_NAME}.json"
EXPORT_DIR = PROJECT_DIR / "export"

CACHE_DIR = Path("/content/piper_cache")
RUN_DIR = Path("/content/training_runs") / VOICE_NAME
CHECKPOINT_CONFIG_PATH = Path("/content/checkpoint_config.yaml")

CACHE_DIR.mkdir(parents=True, exist_ok=True)
RUN_DIR.mkdir(parents=True, exist_ok=True)
EXPORT_DIR.mkdir(parents=True, exist_ok=True)

print("Project dir:", PROJECT_DIR)
print("Metadata:", METADATA_PATH)
print("Audio dir:", AUDIO_DIR)
print("Config path:", CONFIG_PATH)
print("Export dir:", EXPORT_DIR)
print("Run dir:", RUN_DIR)


## 5. Validate the dataset

In [ ]:
import pandas as pd

assert METADATA_PATH.is_file(), f"metadata.csv not found: {METADATA_PATH}"
assert AUDIO_DIR.is_dir(), f"wavs directory not found: {AUDIO_DIR}"

df = pd.read_csv(METADATA_PATH, sep="|", header=None, names=["clip_id", "text"])
assert not df.empty, "metadata.csv is empty"

def resolve_audio_path(clip_id: str) -> Path:
    stem = str(clip_id).strip()
    with_suffix = AUDIO_DIR / stem
    if with_suffix.exists():
        return with_suffix
    return AUDIO_DIR / f"{stem}.wav"

missing = [str(resolve_audio_path(row.clip_id)) for row in df.itertuples() if not resolve_audio_path(row.clip_id).exists()]

print(df.head())
print("Rows:", len(df))
print("Missing audio files:", len(missing))
if missing:
    print("First missing file:", missing[0])

assert not missing, "Some audio files referenced by metadata.csv are missing"


## 6. Write checkpoint config

In [ ]:
checkpoint_config = """trainer:\n  callbacks:\n    - class_path: lightning.pytorch.callbacks.ModelCheckpoint\n      init_args:\n        every_n_epochs: 50\n        save_top_k: 1\n        save_last: true\n"""
CHECKPOINT_CONFIG_PATH.write_text(checkpoint_config, encoding="utf-8")
print(CHECKPOINT_CONFIG_PATH.read_text(encoding="utf-8"))


## 7. Train

In [ ]:
import subprocess

train_cmd = [
    "python3",
    "-m",
    "piper.train",
    "fit",
    "--config",
    str(CHECKPOINT_CONFIG_PATH),
    "--trainer.default_root_dir",
    str(RUN_DIR),
    "--trainer.max_epochs",
    str(MAX_EPOCHS),
    "--data.voice_name",
    VOICE_NAME,
    "--data.csv_path",
    str(METADATA_PATH),
    "--data.audio_dir",
    str(AUDIO_DIR),
    "--model.sample_rate",
    str(SAMPLE_RATE_HZ),
    "--data.espeak_voice",
    ESPEAK_VOICE,
    "--data.cache_dir",
    str(CACHE_DIR),
    "--data.config_path",
    str(CONFIG_PATH),
    "--data.batch_size",
    str(BATCH_SIZE),
]

if BASE_CHECKPOINT.strip():
    train_cmd.extend(["--ckpt_path", BASE_CHECKPOINT.strip()])

print(" ".join(train_cmd))
subprocess.run(train_cmd, cwd="/content/piper1-gpl", check=True)


## 8. Export ONNX from the latest checkpoint

In [ ]:
checkpoints = sorted(RUN_DIR.rglob("*.ckpt"), key=lambda path: path.stat().st_mtime)
assert checkpoints, f"No checkpoints found under {RUN_DIR}"

latest_checkpoint = checkpoints[-1]
onnx_path = EXPORT_DIR / f"{VOICE_NAME}.onnx"

print("Latest checkpoint:", latest_checkpoint)
print("Output ONNX:", onnx_path)


In [ ]:
export_cmd = [
    "python3",
    "-m",
    "piper.train.export_onnx",
    "--checkpoint",
    str(latest_checkpoint),
    "--output-file",
    str(onnx_path),
]

print(" ".join(export_cmd))
subprocess.run(export_cmd, cwd="/content/piper1-gpl", check=True)


## 9. Save export artifacts

In [ ]:
import shutil

assert CONFIG_PATH.is_file(), f"Config json not found: {CONFIG_PATH}"
checkpoint_copy = EXPORT_DIR / latest_checkpoint.name
shutil.copy2(latest_checkpoint, checkpoint_copy)
print("Copied checkpoint:", checkpoint_copy)
print("Export directory contents:")
for path in sorted(EXPORT_DIR.iterdir()):
    print("-", path.name)


## Notes

- This notebook intentionally removes saved outputs and personal hardcoded paths.
- If ONNX export fails for a specific upstream Piper revision, treat that as a version issue and add a clearly labeled workaround cell instead of patching the happy path by default.
- The repo will later add a cleaner local export and `sherpa-onnx` packaging step.